# Cleaned Odyssey's text from Robert Fitzgerald's translation

This is the thirds version. The previous two copies of the text were extremely difficult to clean programmatically. The errata were to varied and copiously disseminated. Case by case, and intance by intance amendment was needed. Because this project focuses on data science not on digital philological work, I decided to find a third copy hoping the prunning can be done using only rule-based methods. This is it (?).

### Before

- Scanned from PDF. Here are some characteristics. 
- Cleanish but
    * Subtitle scheme:  
        `BOOK II`  
        `A HERO'S SON AWAKENS`  
        `[empty line]`
    * Intro and ToC
    * Glossary and notes at the end
    * Separated by books

### After
- Only the text 
- Keep book numbers: 'Book I',…'Book XXIV'
- No numeral digits (if in book title, changed to Roman)

Curiously, Fitzgerald is the **only translator to use "book"** so far. 

In [1]:
translator = "Fitzgerald"
filepath = f"/Users/debr/odysseys_en/raw_txts/Odyssey_{translator}_RAW.txt"

# Define start and end markers
start_marker = "BOOK I"
end_marker = "though still she kept the form and voice of Mentor."

In [2]:
def extract_text_between_markers(file_path, start_marker, end_marker, skip_empty_lines=True):
    """
    Extract text between start and end markers from a text file.
    
    Args:
        file_path (str): Path to the text file
        start_marker (str): Text that marks the beginning of the section to extract
        end_marker (str): Text that marks the end of the section to extract
        skip_empty_lines (bool): Whether to skip empty lines in the output
    
    Returns:
        list: List of strings, each representing a line in the extracted text
    """
    # Read the entire file content
    with open(file_path, "r", encoding="utf-8") as inputfile:
        file_content = inputfile.read()
    
    # Split into lines for exact matching
    all_lines = file_content.splitlines()
    
    # Find the start and end line indices
    start_index = -1 # Initialize with a value that indicates the marker wasn't found
    end_index = -1 # Initialize with a value that indicates the marker wasn't found
    
    for i, line in enumerate(all_lines):
        if line.strip() == start_marker and start_index == -1: # Only find the first occurrence of the start marker
            start_index = i # Mark the first occurrence of the start marker
        elif line.strip() == end_marker and start_index != -1: # Only find the last occurrence of the end marker after the start marker
            end_index = i # Mark the last occurrence of the end marker
            break 
    
    # Handle cases where markers aren't found
    if start_index == -1: # If the start marker isn't found
        print(f"Warning: Exact start marker '{start_marker}' not found in the file.")
        start_index = 0
    
    if end_index == -1:
        print(f"Warning: Exact end marker '{end_marker}' not found in the file.")
        end_index = len(all_lines) - 1
    
    # Extract the lines between markers (inclusive)
    extracted_lines = all_lines[start_index:end_index+1]
    
    # Filter out empty lines if requested
    if skip_empty_lines:
        extracted_lines = [line for line in extracted_lines if line.strip()]
    
    return extracted_lines

def extract_text_between_markers_many_lines(file_path, start_marker, end_marker, skip_empty_lines=True):
    """
    Extract text between start and end markers from a text file.
    
    Args:
        file_path (str): Path to the text file
        start_marker (str): Text that marks the beginning of the section to extract
        end_marker (str): Text that marks the end of the section to extract
        skip_empty_lines (bool): Whether to skip empty lines in the output
    
    Returns:
        list: List of strings, each representing a line in the extracted text
    """
    # Read the entire file content
    with open(file_path, "r", encoding="utf-8") as inputfile:
        file_content = inputfile.read()
    
    # Find the start and end positions
    start_pos = file_content.find(start_marker)
    end_pos = file_content.find(end_marker)
    
    # Handle cases where markers aren't found
    if start_pos == -1:
        print(f"Warning: Start marker '{start_marker}' not found in the file.")
        start_pos = 0
    else:
        # Include the start marker in the output
        start_pos = start_pos
    
    if end_pos == -1:
        print(f"Warning: End marker '{end_marker}' not found in the file.")
        end_pos = len(file_content)
    else:
        # Include the end marker in the output
        end_pos = end_pos + len(end_marker)
    
    # Extract the text between markers
    extracted_text = file_content[start_pos:end_pos]
    
    # Split the extracted text into lines
    lines = extracted_text.splitlines()
    
    # Filter out empty lines if requested
    if skip_empty_lines:
        lines = [line for line in lines if line.strip()]
    
    return lines

In [3]:
# Extract the text One line subtitle
extracted_lines = extract_text_between_markers(filepath, start_marker, end_marker)
# Many lines
# extracted_lines = extract_text_between_markers_manylines(filepath, start_marker, end_marker)

# Verify by printing the end of the extracted text
print(f"Tell me, Python, how {translator}'s Odyssey starts:")
print("\n".join(extracted_lines[:4]))

print(f"\nO, but tell me, Python, how {translator}'s Odyssey ends:")
print("\n".join(extracted_lines[-3:]))

Tell me, Python, how Fitzgerald's Odyssey starts:
BOOK I 
A GODDESS INTERVENES 
Sing in me, Muse, and through me tell the story 
of that man skilled in all ways of contending, 

O, but tell me, Python, how Fitzgerald's Odyssey ends:
set by their arbiter, Athena, daughter 
of Zeus who bears the stormcloud as a shield— 
though still she kept the form and voice of Mentor. 


In [4]:
len(extracted_lines)

13917

In [8]:
def prune_lines_after_pattern(lines_list, pattern, lines_to_prune=[1]):
    """
    Prune specific lines after a pattern match in a list of strings.
    
    Args:
        lines_list (list): List of strings to process
        pattern (str): Pattern to match (e.g., "BOOK I", "BOOK II")
        lines_to_prune (list): List of line numbers to prune after the pattern (0-indexed relative to the pattern)
                               Default is [1] which means prune the line after the pattern
    
    Returns:
        tuple: (pruned_lines, skipped_lines) where:
               - pruned_lines is the list of strings with specified lines removed
               - skipped_lines is a dict mapping skipped line numbers to their content
    """
    result = []
    skip_lines = []
    skipped_content = {}
    
    for i, line in enumerate(lines_list):
        # Check if this line matches the pattern
        if pattern in line:
            result.append(line)  # Keep the pattern line
            
            # Mark lines to skip
            for offset in lines_to_prune:
                if i + offset < len(lines_list):
                    skip_lines.append(i + offset)
                    skipped_content[i + offset] = lines_list[i + offset]
        
        # Skip this line if it's in the skip_lines list
        elif i not in skip_lines:
            result.append(line)
    
    return result, skipped_content

# Example usage:
pruned_lines, skipped_lines = prune_lines_after_pattern(extracted_lines, "BOOK", [1])

# Print verification information
print(f"Pruned {len(skipped_lines)} lines")
print("\nSkipped lines and their content:")
for line_num, content in skipped_lines.items():
    print(f"Line {line_num}: {content[:50]}..." if len(content) > 50 else f"Line {line_num}: {content}")

Pruned 24 lines

Skipped lines and their content:
Line 1: A GODDESS INTERVENES 
Line 503: A HERO’S SON AWAKENS 
Line 967: THE LORD OF THE WESTERN APPROACHES 
Line 1511: THE RED-HAIRED KING AND HIS LADY 
Line 2414: SWEET NYMPH AND OPEN SEA 
Line 2935: THE PRINCESS AT THE RIVER 
Line 3288: GARDENS AND FIRELIGHT 
Line 3661: THE SONGS OF THE HARPER 
Line 4289: NEW COASTS AND POSEIDON’S SON 
Line 4909: THE GRACE OF THE WITCH 
Line 5546: A GATHERING OF SHADES 
Line 6307: SEA PERILS AND DEFEAT 
Line 6889: ONE MORE STRANGE ISLAND 
Line 7442: HOSPITALITY IN THE FOREST 
Line 8075: HOW THEY CAME TO ITHAKA 
Line 8754: FATHER AND SON 
Line 9348: THE BEGGAR AT THE MANOR 
Line 10144: BLOWS AND A QUEEN’S BEAUTY 
Line 10666: RECOGNITIONS AND A DREAM 
Line 11367: SIGNS AND A VISION 
Line 11808: THE TEST OF THE BOW 
Line 12310: DEATH IN THE GREAT HALL 
Line 12878: THE TRUNK OF THE OLIVE TREE 
Line 13302: WARRIORS, FAREWELL 


In [9]:
pruned_lines

['BOOK I ',
 'Sing in me, Muse, and through me tell the story ',
 'of that man skilled in all ways of contending, ',
 'the wanderer, harried for years on end, ',
 'after he plundered the stronghold ',
 'on the proud height of Troy. ',
 'He saw the townlands ',
 'and learned the minds of many distant men, ',
 'and weathered many bitter nights and days ',
 'in his deep heart at sea, while he fought only ',
 'to save his life, to bring his shipmates home. ',
 'But not by will nor valor could he save them, ',
 'for their own recklessness destroyed them all— ',
 'children and fools, they killed and feasted on ',
 'the cattle of Lord Hélios, the Sun, ',
 'and he who moves all day through heaven ',
 'took from their eyes the dawn of their return. ',
 'Of these adventures, Muse, daughter of Zeus, ',
 'tell us in our time, lift the great song again. ',
 'Begin when all the rest who left behind them ',
 'headlong death in battle or at sea ',
 'had long ago returned, while he alone still hungered

In [10]:
len(pruned_lines)

13893

In [12]:
# Because those persistent empty lines, brute force to remove them
filtered_lines  = [line for line in pruned_lines if line.strip()]
len(filtered_lines)

13893

In [13]:
# Extrated lines to text
final_text = "\n".join(filtered_lines)

# Write the cleaned content to a new file for further processing
output_filepath = f"/Users/debr/odysseys_en/cleaned_txts/Odyssey_{translator}_cleaned.txt"
with open(output_filepath, "w", encoding="utf-8") as outputfile:
    outputfile.writelines(final_text)